### SS3 Part B - STADIOalot Order Return Prediction

#### CAP182 - SS3 Part B

This notebook applies the final return-prediction method selected during SS2 to the supplied STADIOalot client data. The final method selected in SS2 was an XGBoost binary classification model. The prediction unit for this stage is an individual order.

The notebook is structured according to the required components:

1. Dataset Loading
2. Dataset Preprocessing
3. Model Training
4. Model Validation
5. Results Visualisation
6. Final Results and Conclusion

### 1. Dataset Loading

The client extract consists of eight related CSV tables covering orders placed between January 2024 and June 2026.
The eight tables are:

1. `table1_orders_transactions.csv`
2. `table2_returns.csv`
3. `table3_product_catalogue.csv`
4. `table4_marketplace_sellers.csv`
5. `table5_customer_information.csv`
6. `table6_customer_behaviour.csv`
7. `table7_inventory_warehouse.csv`
8. `table8_delivery_logistics.csv`

The data is supplied as an uncleaned operational extract in comma-separated values (CSV) format. CSV is suitable for these tabular extracts as it is a widely supported, portable plain-text format that pandas can load directly for inspection, cleaning and analysis. The eight tables are loaded separately first as each represents a different operational subject; they are then checked and joined using the relevant common keys during preprocessing. The agreed prediction unit is an order. An order is labelled as:

- `1` — Returned: at least one return record is associated with the order in the client returns table.
- `0` — Not Returned: no return event is recorded for the order.

The return table is used to construct the target variable however it is not used as a predictive input as its information is only available after the return has occurred.

In [100]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE DISPLAY
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from xgboost import XGBClassifier

import joblib

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 100)

# Make tables easier to read
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

print("Libraries imported successfully.")
# ============================================================
# 2. SET RANDOM SEED
# ============================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"Random seed set to: {RANDOM_STATE}")


Libraries imported successfully.
Random seed set to: 42


In [ ]:
# ============================================================
# 3. DEFINE CLIENT DATA PATH
# ============================================================

# Locate the project root whether the notebook is run from the project
# root or from the notebooks/ directory.
current_path = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        folder
        for folder in [current_path, *current_path.parents]
        if folder.name == "STADIOalot-Return-Prediction"
    ),
    None
)

# Fallback: identify the root by its existing project directories.
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (
            folder
            for folder in [current_path, *current_path.parents]
            if (folder / "data" / "raw" / "ClientData").is_dir()
            and (folder / "experimental_results").is_dir()
        ),
        None
    )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the STADIOalot-Return-Prediction project root. "
        "Run this notebook from the project root or its notebooks folder."
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ClientData"

print("PROJECT ROOT:")
print(PROJECT_ROOT.resolve())

print("\nFULL CLIENT DATA DIRECTORY:")
print(DATA_PATH.resolve())


# Define Client Data Files 
client_filenames = [
    "table1_orders_transactions.csv",
    "table2_returns.csv",
    "table3_product_catalogue.csv",
    "table4_marketplace_sellers.csv",
    "table5_customer_information.csv",
    "table6_customer_behaviour.csv",
    "table7_inventory_warehouse.csv",
    "table8_delivery_logistics.csv"
]
# Display filenames and their full directory paths in a table
file_check = pd.DataFrame({
    "Dataset": client_filenames,
    "Full Directory": [
        str((DATA_PATH / filename).resolve())
        for filename in client_filenames
    ],
    "Status": [
        "FOUND" if (DATA_PATH / filename).is_file()
        else "NOT FOUND"
        for filename in client_filenames
    ]
})

print("\nFULL CLIENT DATA DIRECTORY:")
print(DATA_PATH.resolve())



# Client File Check - Display Full Directory Paths 

# Prevent pandas from truncating long directory paths
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Build the file-check table
file_check = pd.DataFrame({
    "Dataset": client_filenames,
    "Full Directory": [
        str((DATA_PATH / filename).resolve())
        for filename in client_filenames
    ],
    "Status": [
        "FOUND" if (DATA_PATH / filename).is_file()
        else "NOT FOUND"
        for filename in client_filenames
    ]
})

print("\nCLIENT FILE CHECK:")

# Display the complete paths inside the table
display(file_check)

# Validate Client Data Directory and Files 
if not DATA_PATH.is_dir():
    raise FileNotFoundError(
        f"Client data directory not found: {DATA_PATH.resolve()}"
    )

missing_files = [
    filename for filename in client_filenames
    if not (DATA_PATH / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "The following files could not be found:\n"
        + "\n".join(missing_files)
    )

print("\nAll eight client files have been found.")

# ============================================================
# 4. LOAD ALL EIGHT CLIENT TABLES
# ============================================================
table1_orders = pd.read_csv(DATA_PATH / client_filenames[0])
table2_returns = pd.read_csv(DATA_PATH / client_filenames[1])
table3_products = pd.read_csv(DATA_PATH / client_filenames[2])
table4_sellers = pd.read_csv(DATA_PATH / client_filenames[3])
table5_customers = pd.read_csv(DATA_PATH / client_filenames[4])
table6_behaviour = pd.read_csv(DATA_PATH / client_filenames[5])
table7_inventory = pd.read_csv(DATA_PATH / client_filenames[6])
table8_delivery = pd.read_csv(DATA_PATH / client_filenames[7])

print("\nAll eight client tables loaded successfully.")


# ============================================================
# 5. DISPLAY TABLE STRUCTURE
# ============================================================
tables = {
    "Table 1 - Orders and Transactions": table1_orders,
    "Table 2 - Returns": table2_returns,
    "Table 3 - Product Catalogue": table3_products,
    "Table 4 - Marketplace Sellers": table4_sellers,
    "Table 5 - Customer Information": table5_customers,
    "Table 6 - Customer Behaviour": table6_behaviour,
    "Table 7 - Inventory and Warehouse": table7_inventory,
    "Table 8 - Delivery and Logistics": table8_delivery
}

# Summarise the structure of each client dataset
table_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns)
    }
    for name, df in tables.items()
])

print("\nCLIENT DATASET STRUCTURE:")
display(table_summary)

# ============================================================
# 6. DISPLAY COLUMN NAMES IN TABLES
# ============================================================
for name, df in tables.items():

    print(f"\n{name}")

    column_table = pd.DataFrame({
        "Column Number": range(1, len(df.columns) + 1),
        "Column Name": df.columns
    })

    display(column_table)

PROJECT ROOT:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction

FULL CLIENT DATA DIRECTORY:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData

FULL CLIENT DATA DIRECTORY:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData

CLIENT FILE CHECK:


,Dataset,Full Directory,Status
0,table1_orders_transactions.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table1_orders_transactions.csv,FOUND
1,table2_returns.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table2_returns.csv,FOUND
2,table3_product_catalogue.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table3_product_catalogue.csv,FOUND
3,table4_marketplace_sellers.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table4_marketplace_sellers.csv,FOUND
4,table5_customer_information.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table5_customer_information.csv,FOUND
5,table6_customer_behaviour.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table6_customer_behaviour.csv,FOUND
6,table7_inventory_warehouse.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table7_inventory_warehouse.csv,FOUND
7,table8_delivery_logistics.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table8_delivery_logistics.csv,FOUND



All eight client files have been found.

All eight client tables loaded successfully.

CLIENT DATASET STRUCTURE:


,Dataset,Rows,Columns
0,Table 1 - Orders and Transactions,11841,11
1,Table 2 - Returns,2037,11
2,Table 3 - Product Catalogue,11818,11
3,Table 4 - Marketplace Sellers,8241,7
4,Table 5 - Customer Information,8241,11
5,Table 6 - Customer Behaviour,8241,9
6,Table 7 - Inventory and Warehouse,11818,7
7,Table 8 - Delivery and Logistics,8241,10



Table 1 - Orders and Transactions


...

,Column Number,Column Name
0,1,order_id
1,2,customer_id
2,3,look_back_period_days
3,4,sessions
4,5,product_views
5,6,add_to_basket_count
6,7,previous_orders
7,8,previous_returns
8,9,customer_return_rate



Table 7 - Inventory and Warehouse


,Column Number,Column Name
0,1,order_id
1,2,sku
2,3,distribution_centre
3,4,stock_on_hand
4,5,stock_age_days
5,6,historical_pick_pack_time
6,7,inbound_datetime



Table 8 - Delivery and Logistics


,Column Number,Column Name
0,1,order_id
1,2,estimated_delivery_datetime
2,3,delivery_option
3,4,delivery_province
4,5,route_distance_km
5,6,historical_delivery_time_days
6,7,historical_delivery_failure_rate
7,8,actual_delivery_datetime
8,9,delivery_attempts
9,10,on_time_flag


In [ ]:
# ============================================================
# 7. CREATE ORDER-LEVEL TARGET
# ============================================================

# Obtain the distinct list of orders from the main order table.
orders = table1_orders[["order_id"]].drop_duplicates().copy()

# Identify orders with at least one return
returned_orders = (
    table2_returns[["order_id"]]
    .drop_duplicates()
    .assign(Return_Status=1)
)

# Left join returns onto all orders
orders_with_target = orders.merge(
    returned_orders,
    on="order_id",
    how="left"
)

# Orders without a return record are classified as 0.
orders_with_target["Return_Status"] = (
    orders_with_target["Return_Status"]
    .fillna(0)
    .astype(int)
)

print("Order-level target created successfully.")
print(f"Total unique orders: {len(orders_with_target):,}")

In [102]:
# ============================================================
# 8. CLASS DISTRIBUTION AND CLASS IMBALANCE
# ============================================================

# Calculate class counts
class_counts = (
    orders_with_target["Return_Status"]
    .value_counts()
    .sort_index()
)

# Calculate class percentages
class_percentages = (
    orders_with_target["Return_Status"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

# Extract class counts
not_returned = class_counts.get(0, 0)
returned = class_counts.get(1, 0)

# Calculate imbalance ratio
if returned > 0:
    imbalance_ratio = not_returned / returned
else:
    imbalance_ratio = np.nan

# Create combined summary table
class_summary = pd.DataFrame({
    "Class": [
        "0 - Not Returned",
        "1 - Returned",
        "Class Imbalance"
    ],
    "Count": [
        not_returned,
        returned,
        np.nan
    ],
    "Percentage": [
        class_percentages.get(0, 0),
        class_percentages.get(1, 0),
        np.nan
    ],
    "Imbalance Ratio": [
        np.nan,
        np.nan,
        imbalance_ratio
    ]
})

# Display combined table
print("\nCLASS DISTRIBUTION AND IMBALANCE SUMMARY:")

display(
    class_summary.style
    .format({
        "Count": lambda x: f"{x:,.0f}" if pd.notna(x) else "—",
        "Percentage": lambda x: f"{x:.2f}%" if pd.notna(x) else "—",
        "Imbalance Ratio": lambda x: f"{x:.2f}:1" if pd.notna(x) else "—"
    })
    .hide(axis="index")
    .set_caption("Order-Level Target Distribution and Class Imbalance")
)


CLASS DISTRIBUTION AND IMBALANCE SUMMARY:


Class,Count,Percentage,Imbalance Ratio
0 - Not Returned,"6,372",77.32%,—
1 - Returned,"1,869",22.68%,—
Class Imbalance,—,—,3.41:1


### Dataset Loading Summary

The eight client tables have been loaded separately as they contain different types of information linked through keys such as `order_id`, `customer_id`, `seller_id` and `sku`. The modelling dataset contains **8,241 unique orders**, with one row per order, as the business question is to predict whether an order will result in a return.

The target variable is `Return_Status`:

- **Class 0 — Not Returned:** 6,372 orders (**77.32%**).
- **Class 1 — Returned:** 1,869 orders (**22.68%**).

The class imbalance ratio is approximately **3.41:1** (not-returned to returned orders). Returned orders are therefore the minority class. This matters when interpreting performance as a model can achieve seemingly high accuracy by mostly predicting the majority class; precision, recall, F1-score and ROC-AUC are also reported.

**Format justification:** CSV is appropriate for the supplied structured tables as it preserves rows and columns in a portable format and can be read directly by `pandas.read_csv()`. Loading each file separately also allows columns, data types, missing values and join keys to be checked before the tables are combined.

**Target construction:** `table2_returns.csv` is used to identify orders with a return record and construct `Return_Status`; it is not used as a predictive input as it contains information about the outcome itself.